# 02 Silver Cleaning

This notebook creates Silver Delta tables for the inventory management pipeline.

The Silver layer cleans and standardizes raw Bronze data so it can be used for Gold analytics tables.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail_capstone")

spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace| retail_capstone|
+-----------------+----------------+



In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date, round, current_timestamp

In [0]:
bronze_sales_df = spark.table("workspace.retail_capstone.bronze_sales")

silver_sales_df = bronze_sales_df.select(
    col("InvoiceNo").alias("invoice_no"),
    col("StockCode").alias("stock_code"),
    col("Description").alias("product_description"),
    col("Quantity").cast("int").alias("quantity"),
    col("InvoiceDate").alias("invoice_date"),
    col("UnitPrice").cast("double").alias("unit_price"),
    col("CustomerID").cast("double").alias("customer_id"),
    col("Country").alias("country")
).filter(
    col("invoice_no").isNotNull()
).filter(
    col("stock_code").isNotNull()
).filter(
    col("product_description").isNotNull()
).filter(
    col("customer_id").isNotNull()
).filter(
    col("quantity") > 0
).filter(
    col("unit_price") > 0
).filter(
    ~col("invoice_no").startswith("C")
).withColumn(
    "invoice_timestamp",
    to_timestamp(col("invoice_date"))
).withColumn(
    "invoice_date_only",
    to_date(col("invoice_timestamp"))
).withColumn(
    "revenue",
    round(col("quantity") * col("unit_price"), 2)
).withColumn(
    "processed_timestamp",
    current_timestamp()
).dropDuplicates(["invoice_no", "stock_code"])

In [0]:
silver_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_capstone.silver_sales")

In [0]:
display(spark.table("workspace.retail_capstone.silver_sales"))

invoice_no,stock_code,product_description,quantity,invoice_date,unit_price,customer_id,country,invoice_timestamp,invoice_date_only,revenue,processed_timestamp
536370,22544,MINI JIGSAW SPACEBOY,24,2010-12-01T08:45:00.000Z,0.42,12583.0,France,2010-12-01T08:45:00.000Z,2010-12-01,10.08,2026-05-13T13:57:45.669Z
536373,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T09:02:00.000Z,2.55,17850.0,United Kingdom,2010-12-01T09:02:00.000Z,2010-12-01,15.3,2026-05-13T13:57:45.669Z
536373,71053,WHITE METAL LANTERN,6,2010-12-01T09:02:00.000Z,3.39,17850.0,United Kingdom,2010-12-01T09:02:00.000Z,2010-12-01,20.34,2026-05-13T13:57:45.669Z
536378,85183B,CHARLIE & LOLA WASTEPAPER BIN FLORA,48,2010-12-01T09:37:00.000Z,1.25,14688.0,United Kingdom,2010-12-01T09:37:00.000Z,2010-12-01,60.0,2026-05-13T13:57:45.669Z
536378,85071B,RED CHARLIE+LOLA PERSONAL DOORSIGN,96,2010-12-01T09:37:00.000Z,0.38,14688.0,United Kingdom,2010-12-01T09:37:00.000Z,2010-12-01,36.48,2026-05-13T13:57:45.669Z
536381,21934,SKULL SHOULDER BAG,10,2010-12-01T09:41:00.000Z,1.65,15311.0,United Kingdom,2010-12-01T09:41:00.000Z,2010-12-01,16.5,2026-05-13T13:57:45.669Z
536381,15056N,EDWARDIAN PARASOL NATURAL,2,2010-12-01T09:41:00.000Z,5.95,15311.0,United Kingdom,2010-12-01T09:41:00.000Z,2010-12-01,11.9,2026-05-13T13:57:45.669Z
536381,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2010-12-01T09:41:00.000Z,6.75,15311.0,United Kingdom,2010-12-01T09:41:00.000Z,2010-12-01,67.5,2026-05-13T13:57:45.669Z
536382,22839,3 TIER CAKE TIN GREEN AND CREAM,2,2010-12-01T09:45:00.000Z,14.95,16098.0,United Kingdom,2010-12-01T09:45:00.000Z,2010-12-01,29.9,2026-05-13T13:57:45.669Z
536386,85099C,JUMBO BAG BAROQUE BLACK WHITE,100,2010-12-01T09:57:00.000Z,1.65,16029.0,United Kingdom,2010-12-01T09:57:00.000Z,2010-12-01,165.0,2026-05-13T13:57:45.669Z


In [0]:
bronze_products_df = spark.table("workspace.retail_capstone.bronze_products")

silver_products_df = bronze_products_df.select(
    col("stock_code"),
    col("product_name"),
    col("category"),
    col("brand"),
    col("supplier_id"),
    col("unit_cost").cast("double").alias("unit_cost"),
    col("reorder_level").cast("int").alias("reorder_level"),
    col("reorder_quantity").cast("int").alias("reorder_quantity")
).filter(
    col("stock_code").isNotNull()
).filter(
    col("product_name").isNotNull()
).filter(
    col("category").isNotNull()
).filter(
    col("supplier_id").isNotNull()
).filter(
    col("unit_cost") > 0
).filter(
    col("reorder_level") >= 0
).filter(
    col("reorder_quantity") > 0
).dropDuplicates(["stock_code"])

In [0]:
silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_capstone.silver_products")

In [0]:
bronze_inventory_df = spark.table("workspace.retail_capstone.bronze_inventory")

silver_inventory_df = bronze_inventory_df.select(
    col("stock_code"),
    col("warehouse_id"),
    col("current_stock").cast("int").alias("current_stock"),
    col("reserved_stock").cast("int").alias("reserved_stock"),
    to_date(col("last_stock_update")).alias("last_stock_update")
).filter(
    col("stock_code").isNotNull()
).filter(
    col("warehouse_id").isNotNull()
).filter(
    col("current_stock") >= 0
).filter(
    col("reserved_stock") >= 0
).filter(
    col("reserved_stock") <= col("current_stock")
).filter(
    col("last_stock_update").isNotNull()
).withColumn(
    "available_stock",
    col("current_stock") - col("reserved_stock")
).withColumn(
    "processed_timestamp",
    current_timestamp()
).dropDuplicates(["stock_code", "warehouse_id"])

In [0]:
silver_inventory_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_capstone.silver_inventory")

In [0]:
bronze_suppliers_df = spark.table("workspace.retail_capstone.bronze_suppliers")

silver_suppliers_df = bronze_suppliers_df.select(
    col("supplier_id"),
    col("supplier_name"),
    col("supplier_country"),
    col("lead_time_days").cast("int").alias("lead_time_days"),
    col("reliability_score").cast("double").alias("reliability_score")
).filter(
    col("supplier_id").isNotNull()
).filter(
    col("supplier_name").isNotNull()
).filter(
    col("lead_time_days") > 0
).filter(
    (col("reliability_score") >= 0) & (col("reliability_score") <= 1)
).dropDuplicates(["supplier_id"])

In [0]:
silver_suppliers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_capstone.silver_suppliers")

In [0]:
bronze_warehouses_df = spark.table("workspace.retail_capstone.bronze_warehouses")

silver_warehouses_df = bronze_warehouses_df.select(
    col("warehouse_id"),
    col("warehouse_name"),
    col("warehouse_country"),
    col("region"),
    col("capacity").cast("int").alias("capacity")
).filter(
    col("warehouse_id").isNotNull()
).filter(
    col("warehouse_name").isNotNull()
).filter(
    col("region").isNotNull()
).filter(
    col("capacity") > 0
).dropDuplicates(["warehouse_id"])

In [0]:
silver_warehouses_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_capstone.silver_warehouses")

In [0]:
spark.sql("""
SHOW TABLES IN workspace.retail_capstone
""").show(truncate=False)

+---------------+-----------------------+-----------+
|database       |tableName              |isTemporary|
+---------------+-----------------------+-----------+
|retail_capstone|bronze_inventory       |false      |
|retail_capstone|bronze_products        |false      |
|retail_capstone|bronze_sales           |false      |
|retail_capstone|bronze_suppliers       |false      |
|retail_capstone|bronze_warehouses      |false      |
|retail_capstone|curated_online_retail  |false      |
|retail_capstone|customer_summary       |false      |
|retail_capstone|daily_country_revenue  |false      |
|retail_capstone|inventory_sample       |false      |
|retail_capstone|online_retail_source   |false      |
|retail_capstone|product_revenue_summary|false      |
|retail_capstone|products_sample        |false      |
|retail_capstone|raw_online_retail      |false      |
|retail_capstone|silver_inventory       |false      |
|retail_capstone|silver_products        |false      |
|retail_capstone|silver_sale

# Silver Layer Completed

The Silver layer cleans and standardizes data from the Bronze layer.

Created Silver tables:

- `silver_sales`
- `silver_products`
- `silver_inventory`
- `silver_suppliers`
- `silver_warehouses`

Cleaning steps included:

- Column standardization
- Data type casting
- Null filtering
- Invalid record filtering
- Duplicate removal
- Revenue calculation
- Available stock calculation

The Silver tables are now ready for Gold inventory KPI tables.